In [ ]:
import pandas as pd

from Pipeline.Global.GallstoneDataSet import GallstoneDataSet
from Pipeline.Algorithm.ExtremeLearningMachine import ExtremeLearningMachine
from Pipeline.Algorithm.EvaluationMatrix import EvaluationMatrix
from Pipeline.Global.GlobalSetting import GlobalSetting

In [ ]:
model_configs = GlobalSetting.get_model_configs()

In [ ]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_data_path_1()
features_size = gallstone_dataset.x_train_scaled.shape[1]

In [ ]:
x_test_scaled = gallstone_dataset.x_test_scaled
y_test = gallstone_dataset.y_test
x_train_scaled = gallstone_dataset.x_train_scaled
y_train = gallstone_dataset.y_train

In [ ]:
testing_results = []
for config in model_configs:
    for seed in GlobalSetting.elm_initial_state_range:
        elm = ExtremeLearningMachine(
            features_size   = features_size,
            hidden_size     = config["Hidden_Nodes"],
            activation_function     = config["Activation"],
            regularization_lambda   = config["Lambda_Value"]
        )
        elm.initialize_random_weights(random_seed = seed)

        elm.fit(x_train_scaled.values, y_train.values)
        y_pred = elm.predict(x_test_scaled.values)

        evaluation = EvaluationMatrix(y_test, y_pred)
        print(f"--- Config: {config['Hidden_Nodes']} Nodes | Seed: {seed} ---")
        print(evaluation.get_report())

        metrics = evaluation.get_all_metrics()
        test_result = {
            "Model_Type"   : config.get('Model_Types', 'Unknown'),
            "Hidden_Nodes" : config['Hidden_Nodes'],
            "Lambda_Value" : config['Lambda_Value'],
            "Activation"   : config['Activation'].__name__,
            "Seed"         : seed
        }
        test_result.update(metrics)
        testing_results.append(test_result)
        print("\n")

In [ ]:
# 1. 将字典列表转换为 DataFrame
df_results = pd.DataFrame(testing_results)

# 2. 定义哪些列代表一个唯一的 "Algorithm" 配置
group_cols = ["Model_Type", "Hidden_Nodes", "Lambda_Value", "Activation"]

# 3. 动态提取所有需要聚合的指标列（排除配置列和 Seed）
metric_cols = [col for col in df_results.columns if col not in group_cols + ["Seed"]]

# 4. 构造聚合字典：对每个指标求平均值(mean)和标准差(std)
agg_funcs = {metric: ['mean', 'std', 'max' , 'min'] for metric in metric_cols}

# 5. 执行 Group By
summary_df = df_results.groupby(group_cols).agg(agg_funcs).reset_index()

# 将多级表头 (例如 'Accuracy', 'mean') 拼接为单级表头 'Accuracy_mean'
summary_df.columns = [
    f"{col[0]}_{col[1]}" if col[1] else col[0]
    for col in summary_df.columns.values
]

summary_df